# Ball detector full retrain (from COCO-pretrained) - Kaggle

Every prior run (`ball_detector` v1-v6, then the far-court run) fine-tuned on top of the previous `best.pt`. Each generation restarts from an already-converged, low-LR checkpoint and often trains on a narrower slice of data (e.g. the far-court run's `epochs=40` on top of an already-tuned model) - that risks catastrophic forgetting, where the model gets better at whatever the latest run focused on while quietly degrading on everything earlier runs had learned. The far-court checkpoint's poor real-video performance (55-58% raw detection rate across both demo videos) matches that pattern.

This notebook instead starts clean from Ultralytics' COCO-pretrained `yolov8n.pt` and trains once on the **full combined dataset** (all 20,430 train images already accumulated across every prior round, including far-court frames) at `imgsz=1664` (keeps the far-court recall benefit). One joint training run over everything, rather than a chain of sequential fine-tunes.

Same Kaggle usage pattern as `train_ball_detector_farcourt_kaggle.ipynb`:

1. **Dataset comes in as a Kaggle Dataset, not a Drive mount.** Before running this notebook: go to kaggle.com/datasets -> New Dataset -> upload `tennis_ball_dataset_colab.zip` (the same zip already used for prior Colab/Kaggle runs - contains `data/raw/train`, `data/raw/valid`). Then in this notebook, right sidebar -> Add Input -> search for the dataset you just created -> add it.
2. **No Drive-equivalent for continuous checkpoint writes.** `/kaggle/working/` is this notebook's persistent output directory - it survives to the end of a run, but if the interactive session disconnects mid-training you lose progress. For a run this long, use **Save Version -> Save & Run All (Commit)** (top right) instead of running cells interactively - this runs the whole notebook as a background batch job tied to your account (no browser tab needs to stay open), and `/kaggle/working/` is preserved automatically when it finishes.

**Before running:**
- Settings (right sidebar) -> Accelerator -> GPU T4 x2 or GPU P100.
- Settings -> Internet -> **On** (needed for `git clone`, `pip install`, and the automatic `yolov8n.pt` download below - off by default).
- Add the dataset as described above, then run the first cell and check the printed path matches what the rest of this notebook expects - Kaggle's mount folder name comes from your dataset's title, not a fixed path, so `DATASET_DIR` below may need editing.
- This is a full retrain (not a short fine-tune), so budget for a long run - Kaggle GPU sessions are capped at 9-12 hours; `--patience 20` early-stops if validation loss plateaus first, but if it looks likely to run past the cap, lower `--epochs` before committing.

In [ ]:
!ls /kaggle/input/
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# EDIT THIS to match whatever folder the previous cell printed under /kaggle/input/
DATASET_DIR = "/kaggle/input/tennis-ball-dataset-colab"

!find {DATASET_DIR} -maxdepth 3 -type d

In [ ]:
# Clone the repo (public GitHub) at the working branch - needs Settings -> Internet -> On
!git clone -b claude/tennis-ball-yolo-tracking-p8ntwh --depth 1 https://github.com/will-wang1/tennis-tracking-yolo-v1.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

In [ ]:
# Drop the dataset's contents into the repo's expected layout. No weights or
# config to copy this time - configs/ball_dataset.yaml already came from the
# git clone, and the base model below is Ultralytics' COCO-pretrained
# yolov8n.pt (auto-downloaded on first use), not a previous ball_detector.pt.
!rm -rf /kaggle/working/repo/data/raw/train /kaggle/working/repo/data/raw/valid
!mkdir -p /kaggle/working/repo/data/raw
!cp -r {DATASET_DIR}/data/raw/train /kaggle/working/repo/data/raw/train
!cp -r {DATASET_DIR}/data/raw/valid /kaggle/working/repo/data/raw/valid

!ls /kaggle/working/repo/data/raw/train/images | wc -l
!ls /kaggle/working/repo/data/raw/valid/images | wc -l

In [ ]:
# Full retrain from COCO-pretrained yolov8n.pt on the complete dataset.
# imgsz=1664 keeps the far-court recall benefit; epochs=100/patience=20 give
# a from-scratch fine-tune more room to converge than the 40-epoch
# incremental far-court run needed (early stopping will cut it short if
# validation loss plateaus sooner). batch=8 fits a single T4/P100's 16GB at
# this resolution (same reasoning as the far-court notebook - not using the
# second GPU on a T4 x2 instance).
# --project writes into /kaggle/working so it's part of this notebook's
# preserved output - see the markdown intro re: Save & Run All (Commit).
!python scripts/train.py \
    --model yolov8n.pt \
    --imgsz 1664 \
    --epochs 100 \
    --batch 8 \
    --patience 20 \
    --name ball_detector_full_retrain_kaggle \
    --workers 2 \
    --project /kaggle/working/runs

## After training

Once this notebook finishes (or you commit via Save & Run All), the best checkpoint is at `/kaggle/working/runs/ball_detector_full_retrain_kaggle/weights/best.pt`, downloadable from this notebook's **Output** tab (or Data tab, depending on Kaggle's current UI) on kaggle.com. On your local machine:

1. Download `best.pt` from the notebook's output.
2. `cp weights/ball_detector.pt weights/ball_detector_pre_full_retrain_backup.pt` (back up the current one first)
3. Move the downloaded `best.pt` to `weights/ball_detector.pt`
4. Since this was trained at `imgsz=1664`, pass `--imgsz 1664` to `main.py` too when using it - a mismatched inference resolution gives up most of the far-court benefit this was trained for.
5. `python -m pytest tests/ -q` to sanity check nothing broke.
6. Compare validation mAP50-95 against the pre-retrain backup before trusting this over the old checkpoint - this is a from-scratch retrain, not a guaranteed improvement, so verify before committing to it.